# Ensemble Species Distribution Model for *Lupinus polyphyllus* in Norway

**Author:** Hesam Ameri — *MSc Sustainability Management*

An applied machine-learning pipeline that asks a concrete question:

> Where in Norway is the invasive garden lupin (*Lupinus polyphyllus*) climatically suitable today, and how will that change under CMIP6 climate scenarios by 2080?

## Why this project exists
- Demonstrates a **reproducible, end-to-end** geospatial ML pipeline: raw data → predictions → uncertainty maps.
- Uses **ensemble learning** (MaxEnt + Random Forest + Gradient Boosting + Logistic Regression) rather than betting on a single algorithm.
- Employs **spatially-structured cross-validation** (not random k-fold) because SDMs need spatial independence.
- Quantifies **inter-algorithm uncertainty** and **extrapolation risk** (MESS) — two honesty signals that most SDM workflows skip.
- Decomposes future change into **gain vs. loss** to expose that high-emission scenarios actually shrink *net* expansion (thermal overshoot).

## Stack
`Python` · `scikit-learn` · `elapid` (MaxEnt) · `rasterio` · `geopandas` · `cartopy` · `matplotlib`

## Data sources
| Data | Source | License |
|---|---|---|
| Species occurrences | GBIF (Norway subset) | CC-BY |
| Current climate (19 bioclim variables) | WorldClim v2.1 (2.5 arc-min) | CC-BY |
| Elevation → slope | SRTM / Copernicus DEM | open |
| Soil pH | SoilGrids 2.0 | CC-BY |
| Future climate | CMIP6 (ACCESS-CM2, EC-Earth3-Veg, CMCC-ESM2) × (SSP2-4.5, SSP5-8.5) × (2041–2060, 2061–2080) | CC-BY |

## 1. Setup

All configuration is in one place so the pipeline is reproducible.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Point
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import elapid

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------
# Single source of truth for all hyperparameters
# -----------------------------------------------------------------
SEED = 42
N_FOLDS = 4
THRESHOLD = 0.6
MAP_EXTENT = [3, 33, 57.5, 71.5]
MAXENT_FEATURES = ['linear', 'quadratic']   # chosen via prior tuning
MAXENT_BETA = 2.0                            # regularization

DATA_DIR = 'data'
ENV_DIR = os.path.join(DATA_DIR, 'env_layers')
MODEL_DIR = os.path.join(DATA_DIR, 'model')
FIG_DIR = os.path.join(DATA_DIR, 'figures')
FUT_ENS_DIR = os.path.join(DATA_DIR, 'future', 'ensemble_projections')

ALG_COLORS = {'MaxEnt': '#2166ac', 'RF': '#4daf4a',
              'GBM': '#ff7f00', 'GLM': '#984ea3'}
VAR_LABELS = {
    'bio_2': 'Mean diurnal range', 'bio_3': 'Isothermality',
    'bio_8': 'T wettest quarter', 'bio_10': 'T warmest quarter',
    'bio_11': 'T coldest quarter', 'bio_15': 'Precip. seasonality',
    'bio_18': 'Precip. warmest quarter', 'bio_19': 'Precip. coldest quarter',
    'slope': 'Terrain slope', 'soil_ph': 'Soil pH',
}

np.random.seed(SEED)
rng = np.random.RandomState(SEED)

## 2. Load data

- **15,350** spatially-thinned presence points (GBIF, 1-km grid thinning)
- **10,000** KDE-weighted background points (corrects for sampling bias along roads/populated areas)
- **10 predictors**: 8 bioclimatic + slope + soil pH (after multicollinearity screening, VIF < 5)

In [ ]:
with open(os.path.join(ENV_DIR, 'selected_variables_expanded.txt')) as f:
    VARS = [l.strip() for l in f if l.strip() and not l.startswith('#')]

occ = pd.read_csv(os.path.join(DATA_DIR, 'occurrence_env_expanded.csv'))
bg  = pd.read_csv(os.path.join(DATA_DIR, 'background_points_expanded.csv'))
X_occ = occ[VARS].values.astype(np.float64)
X_bg  = bg[VARS].values.astype(np.float64)
X_all = np.vstack([X_occ, X_bg])
y_all = np.concatenate([np.ones(len(X_occ)), np.zeros(len(X_bg))])
print(f'Presences: {len(X_occ):,}   Background: {len(X_bg):,}')
print(f'Predictors ({len(VARS)}): {VARS}')

## 3. Build geographic (spatial) cross-validation folds

Random k-fold leaks information in spatial problems. We cluster points into 4 geographic groups with k-means and hold one out at a time. This is the standard for SDM evaluation.

In [ ]:
all_lons = np.concatenate([occ['lon'].values, bg['lon'].values])
all_lats = np.concatenate([occ['lat'].values, bg['lat'].values])
pts = gpd.GeoSeries([Point(lo, la) for lo, la in zip(all_lons, all_lats)],
                     crs='EPSG:4326')
folds = list(elapid.GeographicKFold(n_splits=N_FOLDS, random_state=SEED).split(pts))
for i, (tr, te) in enumerate(folds):
    print(f'Fold {i+1}: train={len(tr):>6,}  test={len(te):>6,}  '
          f'presence_in_test={(y_all[te]==1).sum():>5,}')

## 4. Define the four algorithms

Each algorithm has a single consistent configuration used everywhere (same across CV, final fit, permutation importance, response curves).

In [ ]:
def make_maxent():
    return elapid.MaxentModel(feature_types=MAXENT_FEATURES,
                               beta_multiplier=MAXENT_BETA,
                               n_hinge_features=10, transform='cloglog')
def make_rf():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                   class_weight='balanced',
                                   random_state=SEED, n_jobs=-1)
def make_gbm():
    return GradientBoostingClassifier(n_estimators=300, max_depth=5,
                                       learning_rate=0.05, min_samples_leaf=10,
                                       subsample=0.8, random_state=SEED)
def make_glm():
    return LogisticRegression(penalty='l2', C=1.0, max_iter=2000,
                               solver='lbfgs', class_weight='balanced',
                               random_state=SEED)

ALG_MAKERS = {'MaxEnt': make_maxent, 'RF': make_rf,
              'GBM': make_gbm, 'GLM': make_glm}

def fit_model(alg, X, y, scaler=None):
    m = ALG_MAKERS[alg]()
    if alg == 'GLM':
        if scaler is None:
            scaler = StandardScaler().fit(X)
        m.fit(scaler.transform(X), y)
        return m, scaler
    m.fit(X, y); return m, None

def predict_model(alg, model, scaler, X):
    if alg == 'MaxEnt':  return model.predict(X)
    if alg == 'GLM':     return model.predict_proba(scaler.transform(X))[:, 1]
    return model.predict_proba(X)[:, 1]

## 5. Spatial cross-validation with corrected Boyce index

Two metrics:
- **AUC** — ranking accuracy (presence vs background).
- **Boyce index** (Hirzel et al. 2006) — whether predictions are *calibrated*: do higher-suitability bins actually contain more presences relative to background? We use the **background** as denominator (standard; some implementations incorrectly use all pixels and inflate the score).

In [ ]:
def boyce_corrected(pred_pres, pred_bg, n_bins=10):
    '''Continuous Boyce: Spearman correlation of (P/E ratio) vs. bin mid.'''
    lo = min(pred_pres.min(), pred_bg.min())
    hi = max(pred_pres.max(), pred_bg.max())
    edges = np.linspace(lo, hi, n_bins + 1)
    pe, centers = [], []
    for i in range(n_bins):
        a, b = edges[i], edges[i+1]
        p_pres = np.mean((pred_pres >= a) & (pred_pres < b))
        p_bg   = np.mean((pred_bg   >= a) & (pred_bg   < b))
        if p_bg > 0:
            pe.append(p_pres / p_bg); centers.append((a+b)/2)
    if len(pe) < 3: return np.nan
    return spearmanr(centers, pe)[0]

In [ ]:
# Run 4-fold spatial CV for all 4 algorithms. Cache out-of-fold predictions
# so we can later build a true held-out ensemble and out-of-fold permutation
# importance.
cv_rows = []
oof_preds = {alg: np.full(len(y_all), np.nan) for alg in ALG_MAKERS}

for alg in ALG_MAKERS:
    print(f'--- {alg} ---')
    for fi, (tr, te) in enumerate(folds):
        m, sc = fit_model(alg, X_all[tr], y_all[tr])
        p_te = predict_model(alg, m, sc, X_all[te])
        oof_preds[alg][te] = p_te
        auc = roc_auc_score(y_all[te], p_te)
        b   = boyce_corrected(p_te[y_all[te]==1], p_te[y_all[te]==0])
        cv_rows.append(dict(algorithm=alg, fold=fi+1, auc=auc, boyce=b))
        print(f'  Fold {fi+1}: AUC={auc:.3f}  Boyce={b:.3f}')

cv_df = pd.DataFrame(cv_rows)
summary = (cv_df.groupby('algorithm')
                .agg(mean_auc=('auc','mean'), sd_auc=('auc','std'),
                     mean_boyce=('boyce','mean'), sd_boyce=('boyce','std'))
                .reset_index().sort_values('mean_auc', ascending=False))
AUC_WEIGHTS = dict(zip(summary['algorithm'], summary['mean_auc']))
summary

**Observation:** MaxEnt and GLM lead (AUC ≈ 0.70). The tree-based models underperform — flexibility without much additional signal, so they overfit. The **ensemble weights by AUC** so weak models contribute less.

## 6. Fit final models on all data and project to Norway

For the production maps we re-fit each algorithm on the full dataset. The ensemble is the AUC-weighted pixel-wise mean — NaN-safe so edge pixels don't drag toward zero.

In [ ]:
# Load env rasters once
ref_path = os.path.join(ENV_DIR, f'norway_{VARS[0]}.tif')
with rasterio.open(ref_path) as ref:
    ref_meta = ref.meta.copy()
    ref_shape = (ref.height, ref.width)
    ref_bounds = ref.bounds
env_rasters = {v: rasterio.open(os.path.join(ENV_DIR, f'norway_{v}.tif')).read(1).astype(np.float64)
               for v in VARS}
valid_mask = np.ones(ref_shape, dtype=bool)
for arr in env_rasters.values():
    valid_mask &= np.isfinite(arr)
valid_rows, valid_cols = np.where(valid_mask)
pixel_data = np.column_stack([env_rasters[v][valid_rows, valid_cols] for v in VARS])
img_extent = [ref_bounds.left, ref_bounds.right, ref_bounds.bottom, ref_bounds.top]
print(f'Valid pixels on the Norway grid: {valid_mask.sum():,}')

In [ ]:
final_models, final_scalers = {}, {}
alg_rasters = {}
for alg in ALG_MAKERS:
    m, sc = fit_model(alg, X_all, y_all)
    final_models[alg] = m; final_scalers[alg] = sc
    p = predict_model(alg, m, sc, pixel_data)
    suit = np.full(ref_shape, np.nan); suit[valid_rows, valid_cols] = p
    alg_rasters[alg] = suit
    print(f'  {alg}: fitted & projected')

# AUC-weighted, NaN-safe ensemble
w = np.array([AUC_WEIGHTS[a] for a in ALG_MAKERS]); w = w / w.sum()
stack = np.stack([alg_rasters[a] for a in ALG_MAKERS], axis=0)
weighted_sum = np.nansum(stack * w[:, None, None], axis=0)
weight_present = np.nansum(np.isfinite(stack) * w[:, None, None], axis=0)
ensemble = np.where(weight_present > 0, weighted_sum / weight_present, np.nan)
ensemble[~valid_mask] = np.nan
inter_sd = np.nanstd(stack, axis=0); inter_sd[~valid_mask] = np.nan
print(f'Ensemble mean suitability over Norway: {np.nanmean(ensemble):.3f}')
print(f'Inter-algorithm mean SD:              {np.nanmean(inter_sd):.3f}')

## 7. Maps

Five panels: each individual algorithm and the ensemble.

In [ ]:
def map_ax(fig, pos):
    ax = fig.add_subplot(*pos, projection=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#e6f2ff', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle='--', color='grey')
    return ax

cmap_suit = cm.get_cmap('cividis').copy(); cmap_suit.set_bad('#e6f2ff')
fig = plt.figure(figsize=(16, 10))
panels = ['MaxEnt','RF','GBM','GLM','Ensemble']
positions = [(2,3,1),(2,3,2),(2,3,3),(2,3,4),(2,3,5)]
for alg, pos in zip(panels, positions):
    ax = map_ax(fig, pos)
    data = alg_rasters[alg] if alg != 'Ensemble' else ensemble
    im = ax.imshow(np.ma.masked_invalid(data), extent=img_extent,
                   origin='upper', transform=ccrs.PlateCarree(),
                   cmap=cmap_suit, vmin=0, vmax=1, interpolation='nearest')
    title = 'Ensemble (AUC-weighted)' if alg == 'Ensemble' else f'{alg} (AUC={AUC_WEIGHTS[alg]:.3f})'
    ax.set_title(title, fontsize=11, fontweight='bold')
cbar_ax = fig.add_axes([0.30, 0.05, 0.4, 0.02])
fig.colorbar(im, cax=cbar_ax, orientation='horizontal', label='Predicted habitat suitability')
fig.suptitle('Habitat suitability for Lupinus polyphyllus in Norway',
             fontsize=14, fontweight='bold', y=0.98)
plt.show()

## 8. Where do algorithms disagree?

The per-pixel SD across MaxEnt / RF / GBM / GLM shows **where the models disagree most**. Transition zones at the current range margin have the highest uncertainty — exactly where a manager would most want to know whether to trust the ensemble.

In [ ]:
cmap_sd = cm.get_cmap('YlOrRd').copy(); cmap_sd.set_bad('#e6f2ff')
fig = plt.figure(figsize=(8, 10)); ax = map_ax(fig, (1,1,1))
im = ax.imshow(np.ma.masked_invalid(inter_sd), extent=img_extent,
                origin='upper', transform=ccrs.PlateCarree(),
                cmap=cmap_sd, vmin=0, vmax=0.3, interpolation='nearest')
plt.colorbar(im, ax=ax, shrink=0.7, label='SD across 4 algorithms')
ax.set_title('Inter-algorithm uncertainty (current climate)', fontsize=12, fontweight='bold')
plt.show()

## 9. Which predictors matter — out-of-fold permutation importance

For each predictor we shuffle its values in the **held-out test fold**, re-predict, and measure the AUC drop. Because the permutation happens on data the model never saw during training, this is a *generalisation* measure — not a training-fit measure. Many published SDM papers use in-sample permutation, which inflates importance.

In [ ]:
def oof_ensemble(oof, weights):
    total = sum(weights.values())
    s = np.zeros(len(next(iter(oof.values()))))
    for alg, p in oof.items():
        s += (weights[alg]/total) * np.nan_to_num(p, nan=0.0)
    return s

base_auc = roc_auc_score(y_all, oof_ensemble(oof_preds, AUC_WEIGHTS))

# Cache fold-trained models (they don't depend on which variable we permute)
fold_models = []
for tr, te in folds:
    fm = {alg: fit_model(alg, X_all[tr], y_all[tr]) for alg in ALG_MAKERS}
    fold_models.append(fm)

N_REPEATS = 5
perm_rows = []
for vi, var in enumerate(VARS):
    drops = []
    for _ in range(N_REPEATS):
        oof_perm = {alg: np.full(len(y_all), np.nan) for alg in ALG_MAKERS}
        for (tr, te), fm in zip(folds, fold_models):
            Xp = X_all[te].copy()
            Xp[:, vi] = rng.permutation(Xp[:, vi])
            for alg in ALG_MAKERS:
                m, sc = fm[alg]
                oof_perm[alg][te] = predict_model(alg, m, sc, Xp)
        drops.append(base_auc - roc_auc_score(y_all, oof_ensemble(oof_perm, AUC_WEIGHTS)))
    perm_rows.append(dict(variable=var, label=VAR_LABELS.get(var, var),
                           mean_auc_drop=np.mean(drops), sd_auc_drop=np.std(drops)))
perm_df = pd.DataFrame(perm_rows).sort_values('mean_auc_drop', ascending=False)
perm_df

In [ ]:
sorted_df = perm_df.sort_values('mean_auc_drop')
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d73027' if d > 0.005 else '#4575b4' for d in sorted_df['mean_auc_drop']]
ax.barh(range(len(sorted_df)), sorted_df['mean_auc_drop'],
         xerr=sorted_df['sd_auc_drop'], capsize=3,
         color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels([VAR_LABELS.get(v, v) for v in sorted_df['variable']])
ax.set_xlabel('Mean decrease in AUC (out-of-fold)')
ax.set_title('Ensemble permutation importance', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
plt.tight_layout(); plt.show()

**Takeaway:** mean temperature of the warmest quarter (bio_10) is by far the most important predictor — a factor of three more than precipitation of the warmest quarter (bio_18). Soil pH, slope, and precipitation seasonality contribute almost nothing at the ensemble level, even though the tree-based models weight them more heavily — a classic illustration of why **algorithm-agnostic** importance is more defensible than any single model's feature_importances_.

## 10. Response curves (partial dependence)

How does predicted suitability respond to each top predictor? We sweep each variable across its 1–99% range while holding all others at their mean and plot one line per algorithm plus the ensemble.

In [ ]:
top6 = perm_df.head(6)['variable'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 9)); axes = axes.flatten()
for i, var in enumerate(top6):
    vi = VARS.index(var); ax = axes[i]
    grid = np.linspace(np.percentile(X_all[:, vi], 1),
                        np.percentile(X_all[:, vi], 99), 100)
    Xg = np.tile(X_all.mean(axis=0), (100, 1)); Xg[:, vi] = grid
    alg_curves = {}
    for alg in ALG_MAKERS:
        p = predict_model(alg, final_models[alg], final_scalers[alg], Xg)
        alg_curves[alg] = p
        ax.plot(grid, p, color=ALG_COLORS[alg], linewidth=1.8, label=alg)
    ens = (np.stack([alg_curves[a] for a in ALG_MAKERS]) * w[:, None]).sum(axis=0)
    ax.plot(grid, ens, color='black', linewidth=2.5, label='Ensemble')
    ax.plot(X_occ[:, vi], np.full(len(X_occ), -0.03), '|',
             color='grey', alpha=0.08, markersize=4)
    ax.set_title(VAR_LABELS.get(var, var), fontweight='bold')
    ax.set_xlabel(VAR_LABELS.get(var, var)); ax.set_ylabel('Suitability')
    ax.set_ylim(-0.06, 1.05)
    if i == 0: ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

**Takeaway:** the response to mean summer temperature (bio_10) is sigmoidal in MaxEnt and GLM but plateaus around ~14 °C in the tree-based models. The ensemble line (black) smooths between them, producing a thermal optimum near 12–14 °C. This matters for the future projections — Norway's southern lowlands cross ~16 °C under SSP5-8.5 by 2080, implying **trailing-edge losses**.

## 11. Future climate projections — gain vs. loss decomposition

Rather than just reporting "% of Norway at high risk in 2080", we decompose the change into:
- **Gain**: currently unsuitable pixels that become suitable
- **Loss**: currently suitable pixels that become unsuitable
- **Net** = Gain − Loss

This exposes a non-intuitive pattern in the projections.

In [ ]:
# We load the already-projected future ensemble rasters (3 GCMs × 2 SSPs × 2 periods)
GCMS = ['ACCESS-CM2', 'EC-Earth3-Veg', 'CMCC-ESM2']
SSPS = ['ssp245', 'ssp585']
PERIODS = ['2041-2060', '2061-2080']
current_binary = (ensemble >= THRESHOLD) & valid_mask
n_valid = int(valid_mask.sum())

rows = []
for ssp in SSPS:
    for per in PERIODS:
        g, l = [], []
        for gcm in GCMS:
            fp = os.path.join(FUT_ENS_DIR, f'ens_{gcm}_{ssp}_{per}.tif')
            if not os.path.exists(fp): continue
            with rasterio.open(fp) as s:
                fsuit = s.read(1)
            fb = (fsuit >= THRESHOLD) & valid_mask
            g.append(((~current_binary) & fb).sum() / n_valid * 100)
            l.append(( current_binary & (~fb)).sum() / n_valid * 100)
        if g:
            rows.append(dict(ssp=ssp, period=per,
                              gain=np.mean(g), gain_sd=np.std(g),
                              loss=np.mean(l), loss_sd=np.std(l),
                              net=np.mean(g) - np.mean(l)))
gl_df = pd.DataFrame(rows); gl_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, ssp in enumerate(SSPS):
    ax = axes[i]
    sub = gl_df[gl_df['ssp']==ssp].reset_index(drop=True)
    x = np.arange(len(sub)); width = 0.35
    ax.bar(x-width/2, sub['gain'], width, yerr=sub['gain_sd'],
            capsize=4, color='#d73027', edgecolor='black', linewidth=0.5, label='Gain')
    ax.bar(x+width/2, sub['loss'], width, yerr=sub['loss_sd'],
            capsize=4, color='#4575b4', edgecolor='black', linewidth=0.5, label='Loss')
    ax.set_xticks(x); ax.set_xticklabels(sub['period'])
    ax.set_ylabel("% of Norway's land area")
    ax.set_title('SSP2-4.5' if ssp=='ssp245' else 'SSP5-8.5', fontweight='bold')
    for j, r in sub.iterrows():
        ax.text(j, max(r['gain'], r['loss']) + 2, f'Net: {r["net"]:+.1f}%',
                 ha='center', fontweight='bold', fontsize=9)
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Counter-intuitive finding:** under the **stronger-warming** SSP5-8.5, the *net* expansion is **smaller** than under SSP2-4.5 by late century (+13.9% vs +16.8%). Gain is roughly constant (~25% of land area) but **losses more than double** as southern lowlands exceed the species' thermal optimum. This is **trailing-edge contraction** — a mechanism rarely quantified in invasive-species SDMs.

## 12. Honesty check — MESS extrapolation map

All SDMs extrapolate into climate space they haven't been trained on. The **MESS** (Multivariate Environmental Similarity Surface) flags pixels where at least one predictor is outside the training range — predictions there are unreliable.

In [ ]:
train_min = X_all.min(axis=0); train_max = X_all.max(axis=0)
per_var_mess = np.full((len(VARS),) + ref_shape, np.nan)
for vi, var in enumerate(VARS):
    arr = env_rasters[var]
    lo, hi = train_min[vi], train_max[vi]
    below = arr < lo; above = arr > hi; inside = ~below & ~above
    score = np.full(ref_shape, np.nan)
    score[inside] = 100
    score[below]  = 100 * (arr[below] - lo) / max(hi - lo, 1e-9)
    score[above]  = 100 * (hi - arr[above]) / max(hi - lo, 1e-9)
    per_var_mess[vi] = score
mess = np.nanmin(per_var_mess, axis=0); mess[~valid_mask] = np.nan
print(f'Pixels outside training envelope: {(mess<0).sum()/n_valid*100:.1f}% of Norway')

cmap_mess = cm.get_cmap('RdYlGn').copy(); cmap_mess.set_bad('#e6f2ff')
fig = plt.figure(figsize=(8, 10)); ax = map_ax(fig, (1,1,1))
im = ax.imshow(np.ma.masked_invalid(mess), extent=img_extent,
                origin='upper', transform=ccrs.PlateCarree(),
                cmap=cmap_mess, vmin=-50, vmax=100, interpolation='nearest')
plt.colorbar(im, ax=ax, shrink=0.7, label='MESS score')
ax.set_title('MESS: negative = extrapolation (current climate)',
              fontsize=12, fontweight='bold')
plt.show()

## Summary

| Metric | Value |
|---|---|
| Spatial-CV AUC (best single: MaxEnt) | **0.70** ± 0.03 |
| Spatial-CV AUC (worst single: GBM) | 0.63 ± 0.04 |
| Spatial-CV Boyce (corrected) | 0.90 – 0.98 |
| Dominant predictor (OOF perm.) | **Mean T warmest quarter (bio_10)**, ΔAUC = 0.15 |
| Current area at risk (≥ 0.6) | ~20% of Norway |
| Net change by 2080 under SSP5-8.5 | **+14%** (25% gain, 12% loss) |
| Net change by 2080 under SSP2-4.5 | +17% (25% gain, 9% loss) |
| Pixels outside training envelope | ~40% (MESS < 0) |

### What this project demonstrates as a portfolio piece

1. **End-to-end ML pipeline** on real geospatial data: from GBIF download → cleaning → bias correction → modelling → uncertainty quantification → future projection.
2. **Spatial-aware cross-validation** (not random k-fold).
3. **Algorithm-agnostic evaluation**: AUC + corrected Boyce index + per-algorithm uncertainty maps.
4. **Out-of-fold permutation importance** — defensible variable importance that doesn't depend on any single model's internal weighting.
5. **Honest uncertainty reporting**: inter-algorithm SD map + MESS extrapolation map.
6. **Decision-relevant output**: gain/loss decomposition reveals thermal overshoot — a finding that a simple "% area at risk" summary would hide.

### Known limitations

- Moderate discrimination (AUC ≈ 0.70) — specific pixel-level predictions are uncertain.
- No dispersal or land-use predictors → the maps describe the *climatic envelope*, not where the species will actually spread by a given year.
- ~40% of Norway's land area lies outside the training envelope (MESS < 0), so predictions there should be treated as extrapolations.
- Single-GCM variance within each SSP is substantial; multi-GCM means smooth this but cannot eliminate it.

---

## Appendix A — Try it on your own Norwegian invasive

The pipeline above is tuned for *Lupinus polyphyllus*, but the modelling machinery is species-agnostic. The helper module `quicklook.py` wraps a fast subset of the pipeline (~3–5 minutes) that anyone can run on any GBIF-listed species in Norway:

**GBIF download → cleaning + 1-km thinning → env extraction → KDE bias-corrected background → MaxEnt + GLM with 4-fold spatial CV → ensemble suitability map.**

It deliberately skips the parts of the full pipeline that would slow a demo to a crawl or fail for species with sparse data: Random Forest, Gradient Boosting, CMIP6 future projections, and MESS. Use `portfolio_pipeline.py` if you want all of those.

**Guardrails:**
- Needs ≥ 200 cleaned occurrences (below that, spatial CV is unreliable).
- Caps at 50,000 records (keeps the GBIF download under a few minutes).
- Prints a clear error if the species doesn't pass these checks so there's no confusing traceback.

Try one of the suggested species below, or type any binomial you like.

In [ ]:
from quicklook import run_quicklook, EXAMPLE_SPECIES

print("Suggested invasive species (well-sampled in Norway):")
for s in EXAMPLE_SPECIES:
    print(f"  - {s}")

### Pick a species and run

Change `SPECIES` below to any binomial you like (e.g. `"Heracleum mantegazzianum"`, `"Impatiens glandulifera"`, `"Rosa rugosa"`). The cell will download occurrences, fit the quick-look ensemble, and render a suitability map with CV scores in the title.

In [ ]:
# Change this line to try a different species
SPECIES = "Heracleum mantegazzianum"   # giant hogweed

result = run_quicklook(SPECIES, country="NO", n_bg=5000, verbose=True)

print("\n=== CV summary ===")
print(result["cv_summary"].to_string(index=False))
print(f"\nOccurrences used: {result['n_occurrences']:,}")

### Interpreting the quick-look output

- **AUC** below ~0.65 → the model is barely better than guessing; predictions should be taken as exploratory.
- **AUC 0.65 – 0.75** → moderate — acceptable for ranking areas by relative risk, but avoid over-interpreting exact pixels. This is where *Lupinus polyphyllus* sits.
- **AUC > 0.80** → strong; you likely have a species with a narrow, well-sampled climatic niche.
- **Boyce < 0.5** → the model ranks points decently (AUC) but predictions are *miscalibrated*; a higher suitability score doesn't reliably mean higher occurrence density.

Common reasons a species fails the ≥ 200 occurrence threshold:
1. It's rare or restricted to one region.
2. It's cultivated (GBIF filters out most but not all).
3. The Latin binomial has a typo — check `Lupinus polyphyllus` style (genus + species, space, no authority).

**Want the full treatment?** Rerun `portfolio_pipeline.py` after swapping the occurrence CSV — it will produce permutation importance, response curves, MESS, and CMIP6 future projections for any species.